In [1]:
# import kagglehub
# import shutil
# import os

# # Download latest version (this goes into KaggleHub cache folder)
# path = kagglehub.dataset_download("kazanova/sentiment140")

# print("KaggleHub download path:", path)

# # Your custom target folder
# custom_path = r"E:\github\data science\data-science\NLP"

# # Make sure the folder exists
# os.makedirs(custom_path, exist_ok=True)

# # Copy everything from KaggleHub cache folder to your custom folder
# for item in os.listdir(path):
#     s = os.path.join(path, item)
#     d = os.path.join(custom_path, item)
#     if os.path.isdir(s):
#         shutil.copytree(s, d, dirs_exist_ok=True)
#     else:
#         shutil.copy2(s, d)

# print("Dataset copied to:", custom_path)


In [2]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.optim import Adam

import matplotlib.pyplot as plt
import seaborn as sns
import time

In [3]:
df=pd.read_csv(r"E:\github\data science\data-science\NLP\training.1600000.processed.noemoticon.csv",encoding='latin-1',header=None)
df.columns=['target','ids','date','flag','user','text']
df.fillna('')
print(df.head())
print(df.describe())
print(df.info())

   target         ids                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  
             target           ids
count  1.600000e+06  1.600000e+06
mean   2.000000e+00  1.998818e+09
std    2.000001e+00  1.935761e+08
min    0.000000e+00  1.467810e+09
25%    

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import re
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nltk
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
print(stop_words)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


{'own', 'their', "you'd", 'yourselves', 'on', 'too', "we'd", 'against', 'mightn', 'weren', 'y', 'being', 'we', "she'll", "that'll", 'yourself', 'now', 're', 'can', 'a', 'to', 'ours', 'should', 'while', 'her', 'here', "she'd", 'after', "you're", 'mustn', 'shan', 'any', 'further', 'have', 'through', 'up', "it'd", 'until', 'has', 'himself', "we'll", 'been', 'what', 'other', 'by', 'down', 'nor', 'who', 'an', "couldn't", 't', 'hasn', 'only', "shan't", 'once', "i'd", "they'd", 'ain', 'these', 'it', 'off', 'of', 'its', 'm', "needn't", "weren't", 'very', 'are', 'is', 'how', "i've", 'them', 'your', 'such', 'my', 'between', "hadn't", 'having', 'won', 'but', "she's", 'ma', 'all', 'from', 'they', 've', 'herself', 'out', 'd', "we're", 'both', 'shouldn', 'about', 'into', 'was', 'as', "isn't", 'me', 'because', 'where', "shouldn't", "they're", 'didn', 'hadn', 'wasn', 'our', 'when', "you've", 'am', 'above', 'those', "they'll", 'don', "haven't", "i'll", 'doesn', 'just', 'there', 'then', 'which', 'that',

[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


0->negative
1->positive


In [5]:
port_stem = PorterStemmer()
def stemming_tokenizer(text):
    stem_content=re.sub('[^a-zA-Z]',' ',text)
    stem_content=stem_content.lower()
    stem_content=stem_content.split()
    stem_content=[port_stem.stem(word) for word in stem_content if not word in stop_words]
    stem_content=' '.join(stem_content)
    return stem_content
print(f'example:{stemming_tokenizer("This is a sample text. We\'re testing the stemming_tokenizer function!")}')

example:sampl text test stem token function


In [6]:

df.loc[:99999, 'stemmed_text'] = df.loc[:99999, 'text'].apply(stemming_tokenizer)


In [7]:
df['stemmed_text'] = df['text'].apply(stemming_tokenizer)
print(df.head(3))
print(df['stemmed_text'])


   target         ids                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   

              user                                               text  \
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1    scotthamilton  is upset that he can't update his Facebook by ...   
2         mattycus  @Kenichan I dived many times for the ball. Man...   

                                        stemmed_text  
0  switchfoot http twitpic com zl awww bummer sho...  
1  upset updat facebook text might cri result sch...  
2  kenichan dive mani time ball manag save rest g...  
0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi fee

In [8]:
df['target'] = df['target'].replace(4, 1)

In [9]:
x=df['stemmed_text'].values
y=df['target'].values
print(x)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [10]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
vectorizer=TfidfVectorizer()
x_train=vectorizer.fit_transform(x_train)
x_test=vectorizer.transform(x_test)
print(x_train.shape,x_test.shape)
print(x_train)
print(x_test)

(1280000, 460826) (320000, 460826)
  (0, 191047)	0.4158134532708448
  (0, 451664)	0.17407838168369683
  (0, 329290)	0.13943539710756206
  (0, 234371)	0.11547791834728915
  (0, 306938)	0.2540227006678372
  (0, 321926)	0.23170650142956095
  (0, 413565)	0.38402205228527403
  (0, 67106)	0.2573367544771409
  (0, 334073)	0.40408019307479154
  (0, 452952)	0.15865049735043385
  (0, 376651)	0.1690998567396651
  (0, 150596)	0.1126574366703869
  (0, 37131)	0.21316028397270034
  (0, 334025)	0.16640577147832103
  (0, 322206)	0.35950313299806835
  (1, 130482)	0.3794466709590718
  (1, 112840)	0.4696148165894244
  (1, 5661)	0.3481704011758972
  (1, 358026)	0.31128931514918085
  (1, 121045)	0.6460328799347043
  (2, 234371)	0.225452958825765
  (2, 346845)	0.673827846960868
  (2, 364531)	0.41392230143649633
  (2, 234455)	0.5690301612307586
  (3, 322565)	0.24777499292841212
  :	:
  (1279997, 332674)	0.39846363367368987
  (1279997, 28565)	0.5192441258938135
  (1279997, 324483)	0.7560504416798717
  (1279998

In [11]:
# logistic regression
model=LogisticRegression(max_iter=100000,class_weight='balanced')
model.fit(x_train,y_train)
x_train_pred=model.predict(x_train)

In [12]:
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.7736125
              precision    recall  f1-score   support

           0       0.79      0.75      0.77    159494
           1       0.76      0.80      0.78    160506

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000

[[119497  39997]
 [ 32447 128059]]


In [13]:
import pickle
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

texts = [
    "I love this product",
    "This is the worst experience ever",
    "Not bad, could be better",
    "Excellent customer service and fast delivery"
]
labels = [1, 0, 1, 1] 

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)
X = vectorizer.fit_transform(texts)

# Train classifier
model = LogisticRegression()
model.fit(X, labels)

# Save both model and vectorizer
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved!")


Model and vectorizer saved!


In [15]:
# Load model and vectorizer
with open('sentiment_model.pkl', 'rb') as file:
    model = pickle.load(file)

with open('vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

# Simulate input text
text = "This product is great!"

# Your preprocessing function here (match training preprocessing)
processed_text = text.lower()

# Vectorize
vectorized_text = vectorizer.transform([processed_text])

# Predict
prediction = model.predict(vectorized_text)[0]
print(f"Prediction: {prediction}")


Prediction: 1
